# Lecture 8 · Exercises

Code for [exercises.md](exercises.md)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

DATA = Path("../../coding_tasks/Data/Out_Data")
rng = np.random.default_rng(8)

Two helpers used throughout. The first rebuilds the ANOVA table from scratch so the decomposition is visible rather than taken on trust, the second runs every pairwise comparison and applies the Bonferroni threshold.

In [ ]:
def anova_by_hand(frame, value, group):
    g = frame.groupby(group)[value]
    grand = frame[value].mean()
    n, k = len(frame), g.ngroups
    ssg = sum(len(v) * (v.mean() - grand) ** 2 for _, v in g)
    sse = sum((len(v) - 1) * v.var(ddof=1) for _, v in g)
    msg, mse = ssg / (k - 1), sse / (n - k)
    f = msg / mse
    return pd.Series(
        {
            "SSG": ssg,
            "SSE": sse,
            "SST": ssg + sse,
            "df_G": k - 1,
            "df_E": n - k,
            "MSG": msg,
            "MSE": mse,
            "F": f,
            "p": stats.f.sf(f, k - 1, n - k),
            "eta_sq": ssg / (ssg + sse),
        }
    )


def pairwise(frame, value, group, mse=None, df_e=None):
    col = frame[group]
    names = list(col.cat.categories) if isinstance(col.dtype, pd.CategoricalDtype) else sorted(col.unique())
    rows = []
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            x, y = frame[value][frame[group] == a], frame[value][frame[group] == b]
            if mse is None:
                r = stats.ttest_ind(x, y, equal_var=False)
                t, dof, p = r.statistic, r.df, r.pvalue
            else:
                se = np.sqrt(mse * (1 / len(x) + 1 / len(y)))
                t, dof = (x.mean() - y.mean()) / se, df_e
                p = 2 * stats.t.sf(abs(t), dof)
            rows.append({"pair": f"{a} vs {b}", "diff": x.mean() - y.mean(), "t": t, "df": dof, "p": p})

    k = frame[group].nunique()
    alpha_star = 0.05 / (k * (k - 1) / 2)
    out = pd.DataFrame(rows).set_index("pair")
    out["bonferroni p"] = (out["p"] * len(out)).clip(upper=1)
    out["reject"] = out["p"] < alpha_star
    return out, alpha_star

---

## Exercise 8.1 · Teaching methods

No dataset, only the printed ANOVA table. What is worth doing is rebuilding it: every number follows from the two sums of squares and the two degrees of freedom, and the within-group standard deviation is the one figure the table does not print but the plot can be checked against.

In [ ]:
table = pd.DataFrame({"sum_sq": [97.36, 143.06], "df": [4, 40]}, index=["method", "Residuals"])
table["mean_sq"] = table["sum_sq"] / table["df"]

F = table.loc["method", "mean_sq"] / table.loc["Residuals", "mean_sq"]
print(table.round(3))
print(f"\nF = {F:.3f}   p = {stats.f.sf(F, 4, 40):.6f}")
print(f"critical value at 5% = {stats.f.ppf(0.95, 4, 40):.3f}, at 1% = {stats.f.ppf(0.99, 4, 40):.3f}")
print(f"within-group sd = {np.sqrt(table.loc['Residuals', 'mean_sq']):.2f}")

Reject H₀ at p = 0.000282. The five teaching methods do not all produce the same average score, and the three that involve lectures sit above the two pure self-study formats.

A within-group sd of 1.89 matches the scatter on the plot, where most groups span about four points. The eye and the table agree.

What the F-test has **not** told us is which methods differ. With five groups that is ten comparisons, and the uncorrected error rate grows accordingly.

In [ ]:
fwer = pd.DataFrame({"groups": range(3, 7)})
fwer["comparisons"] = fwer["groups"] * (fwer["groups"] - 1) // 2
fwer["alpha star"] = 0.05 / fwer["comparisons"]
fwer["P(at least one false positive)"] = 1 - 0.95 ** fwer["comparisons"]
fwer.set_index("groups").round(4)

---

## Exercise 8.2 · Pain relief

Amundson et al. (2017), three treatments for pain after knee surgery, 150 patients. Higher scores mean more pain, so the best treatment is the one with the lowest mean.

In [ ]:
pain = pd.read_csv(DATA / "lecture8_painstudy.csv")
print(pain.shape, "| missing:", pain.isna().sum().sum())
print(pain.groupby("trt")["pain"].agg(["size", "mean", "std", "min", "max"]).round(4))

### 1 · the distribution across groups

The outcome is an integer from 1 to 10, so 50 points per group would stack into ten positions without jitter. Horizontal jitter spreads them out, vertical jitter would invent values nobody reported.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
colours = {"A": "#4C78A8", "B": "#B03A2E", "C": "#2E8B57"}

for i, name in enumerate(["A", "B", "C"]):
    y = pain.pain[pain.trt == name]
    ax.scatter(i + rng.uniform(-0.18, 0.18, len(y)), y, s=22, alpha=0.65, color=colours[name])
    ax.hlines(y.mean(), i - 0.3, i + 0.3, color="black", lw=2)

ax.set_xticks(range(3), ["Treatment A", "Treatment B", "Treatment C"])
ax.set_ylabel("maximum pain score, day 1 (higher is worse)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Group C sits clearly below the other two. A and B are closer together, with B slightly worse.

### 2, 3, 4 · means and the ANOVA

H₀: μ_A = μ_B = μ_C. H_A: at least two differ.

In [ ]:
print(pain.groupby("trt")["pain"].mean().round(2).to_string())
print()
print(anova_lm(smf.ols("pain ~ C(trt)", data=pain).fit(), typ=2).round(4))
print()
print(anova_by_hand(pain, "pain", "trt").round(4).to_string())
print(f"\nF critical at 5% = {stats.f.ppf(0.95, 2, 147):.4f}")

Reject H₀. F = 98.92 against a critical value of 3.06, and the p-value is below what floating point prints.

η² = 0.574, so 57% of the total variation in pain scores is accounted for by which treatment the patient received. For a clinical outcome that is very large.

### 5 · pairwise comparisons with the Bonferroni correction

K = 3 comparisons, so α* = 0.05 / 3 = 0.01667.

In [ ]:
result, alpha_star = pairwise(pain, "pain", "trt")
print(f"alpha* = {alpha_star:.5f}")
result.round(6)

All three nulls are rejected. The ordering **C < A < B** is fully resolved, which is more than the ANOVA alone could say.

Note how far clear the p-values are: multiplying each by K changes nothing, because every one survives by orders of magnitude. The correction only bites on p-values near the threshold. Compare the income example further down, where the same correction on the same number of comparisons flips two of three decisions.

Two assumptions worth checking rather than assuming. The official solution uses Welch t-tests while `aov` pools a single MSE across all three groups, which is inconsistent in principle. Three variants, to see what the choice costs.

In [ ]:
rows = []
for a, b in [("A", "B"), ("A", "C"), ("B", "C")]:
    x, y = pain.pain[pain.trt == a], pain.pain[pain.trt == b]
    welch = stats.ttest_ind(x, y, equal_var=False)
    pooled2 = stats.ttest_ind(x, y, equal_var=True)
    t3 = (x.mean() - y.mean()) / np.sqrt(1.7986 * (1 / len(x) + 1 / len(y)))
    rows.append(
        {
            "pair": f"{a} vs {b}",
            "welch t": welch.statistic,
            "2-sample pooled t": pooled2.statistic,
            "all-groups MSE t": t3,
            "welch p": welch.pvalue,
            "2-sample pooled p": pooled2.pvalue,
            "all-groups MSE p": 2 * stats.t.sf(abs(t3), 147),
        }
    )

variants = pd.DataFrame(rows).set_index("pair")
print(variants[["welch t", "2-sample pooled t", "all-groups MSE t"]].round(4))
print()
print(variants[["welch p", "2-sample pooled p", "all-groups MSE p"]].round(8))
print()
print(stats.levene(*[v.values for _, v in pain.groupby("trt")["pain"]], center="mean"))

Welch and the two-sample pooled t are identical to four decimals, because the groups are the same size, which makes the two formulas algebraically the same. Only the degrees of freedom move, 91.6 to 96.7 instead of 98.

The third variant is a different thing. It pools MSE across all 150 patients rather than the 100 in each comparison, so the t-statistic itself changes, and it does not uniformly help: **A vs B gets a larger p**, 0.000051 against 0.000020, because group C's variance of 2.28 inflates the pooled estimate used for a comparison C is not part of. A vs C and B vs C get smaller ones from the extra degrees of freedom. Pooling pays when the variances really are equal and taxes the tighter comparisons when they are not.

Levene gives p = 0.068, so equal variances are not rejected at the 5% level, though group C's sd of 1.51 against B's 1.15 is why it is borderline. No variant changes a decision here.

---

## Beyond the exercise · the lecture's own examples

### Example 8.1 · income and work effort

27 students, three income groups. The group column is `work` with values **T, M, H**: `T` is German *tief* and is the slides' "Low". `work_order` exists only to force the plotting order.

In [ ]:
work = pd.read_csv(DATA / "lecture8_income_work_effort.csv")
work["income"] = pd.Categorical(
    work["work"].map({"T": "Low", "M": "Medium", "H": "High"}),
    categories=["Low", "Medium", "High"],
    ordered=True,
)

print(work.groupby("income", observed=True)["work_effort"].agg(["size", "mean", "var"]).round(4))
print("overall mean", round(work.work_effort.mean(), 4))
print()
print(anova_by_hand(work, "work_effort", "income").round(4).to_string())

SSG + SSE = SST exactly, and F = 11.126 on (2, 24) with p = 0.00038 reproduces the slide's R output.

The slide's graph, with a black bar at each group mean and a dashed line at the overall mean. Between-group variation is how far the bars sit from the dashed line, within-group variation is the scatter around each bar, and the F-statistic is the ratio.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))

for i, name in enumerate(["Low", "Medium", "High"]):
    y = work.work_effort[work.income == name]
    ax.scatter(i + rng.uniform(-0.12, 0.12, len(y)), y, s=28, alpha=0.8, color="#4C78A8")
    ax.hlines(y.mean(), i - 0.28, i + 0.28, color="black", lw=2)

ax.axhline(work.work_effort.mean(), ls="--", color="#B03A2E", lw=1.2)
ax.set_xticks(range(3), ["Low", "Medium", "High"])
ax.set_xlabel("Income")
ax.set_ylabel("Work Effort")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### Slide 15 does not reconcile with this dataset

The Bonferroni slide lists three p-values, 0.003, 0.00003 and 0.85, and labels the third comparison μ_L = μ_H, repeating the second. The label is a typo for μ_M = μ_H. The numbers are the bigger problem: no standard pairwise test produces them.

In [ ]:
pairs = [("Low", "Medium"), ("Low", "High"), ("Medium", "High")]
index = [f"{a} vs {b}" for a, b in pairs]

tukey = pairwise_tukeyhsd(work.work_effort, work.income)
tukey_p = {}
for g1, g2, _, p_adj, *_ in tukey.summary().data[1:]:
    tukey_p[(g1, g2)] = tukey_p[(g2, g1)] = p_adj

compare = pd.DataFrame(
    {
        "two-sample, Welch": [
            stats.ttest_ind(work.work_effort[work.income == a], work.work_effort[work.income == b],
                            equal_var=False).pvalue
            for a, b in pairs
        ],
        "two-sample, pooled": [
            stats.ttest_ind(work.work_effort[work.income == a], work.work_effort[work.income == b],
                            equal_var=True).pvalue
            for a, b in pairs
        ],
        "pooled MSE, df = 24": pairwise(work, "work_effort", "income", mse=1.1019, df_e=24)[0]["p"].reindex(index).values,
        "Tukey HSD": [tukey_p[(a, b)] for a, b in pairs],
        "slide 15 claims": [0.003, 0.00003, 0.85],
    },
    index=index,
)
compare.round(6)

Nothing lands near 0.003, and nothing comes close to 0.85. Groups M and H have means 5.00 and 6.22 with variances near 1, so no test is going to call them indistinguishable. The ANOVA output on slide 11 reproduces exactly, so the dataset is right and slide 15 is wrong.

What the data actually say, at α* = 0.01667:

In [ ]:
actual, alpha_star = pairwise(work, "work_effort", "income")
print(f"alpha* = {alpha_star:.5f}")
actual.round(6)

Only **Low versus High** survives the correction. The other two would both have been significant at an uncorrected 5% level and both lose it, Medium vs High only barely at 0.0182 against 0.01667.

That is a better teaching example than the slide's version. The ANOVA rejects decisively at p = 0.00038, and after correcting for three comparisons the only pair that can be separated individually is the two extremes.

### Example 8.2 · four datasets, four F-values

The within-group spread is about the same in all four panels, so the ranking is decided entirely by how far apart the group means sit. Reading the means off the panels and assuming a within-group sd of 1 reproduces the ranking and roughly the magnitudes.

In [ ]:
panels = {
    "Dataset 1": [10, 10, 10, 10, 10],
    "Dataset 2": [10, 12.5, 15.5, 19, 22],
    "Dataset 3": [10, 11, 12.5, 14.5, 15.5],
    "Dataset 4": [10, 10.5, 11, 11.5, 11.5],
}

rows = []
for name, means in panels.items():
    m = np.array(means, dtype=float)
    rows.append(
        {
            "panel": name,
            "spread of means": m.max() - m.min(),
            "MSG with n = 10 per group": 10 * ((m - m.mean()) ** 2).sum() / 4,
        }
    )

out = pd.DataFrame(rows).set_index("panel")
out["F if sd = 1"] = out["MSG with n = 10 per group"]
out["assigned"] = [0.95, 290.9, 56.69, 4.38]
out.round(2)

Two things the panels teach. F near 1 is what no effect looks like, because under H₀ both mean squares estimate the same variance. And Dataset 4, with F = 4.38 on (4, 45), has p ≈ 0.004 despite the groups overlapping heavily on the plot: a visible difference and a detectable difference are not the same thing.

### Example 8.3 · exam date and academic performance

Mouritsen, Davis and Jones (2016). The conjecture was that students taking the exam later do better, through information leakage.

In [ ]:
exam = pd.DataFrame(
    {
        "n": [174, 207, 338, 738],
        "mean score": [0.852935, 0.822888, 0.799981, 0.713808],
        "mean gpa": [3.32, 3.16, 3.18, 2.96],
    },
    index=pd.Index([1, 2, 3, 4], name="day"),
)

ssg, sse, df_g, df_e = 4.354, 36.459, 3, 1453
msg, mse = ssg / df_g, sse / df_e
print(f"MSG = {msg:.4f} (table 1.451), MSE = {mse:.5f} (table .025)")
print(f"F   = {msg / mse:.3f} (table 57.835), p = {stats.f.sf(msg / mse, df_g, df_e):.2e}")
print(f"n = {exam['n'].sum()}, total df = {df_g + df_e}")
print(f"correlation between day mean score and day mean GPA: {exam['mean score'].corr(exam['mean gpa']):.4f}")
exam

The table is internally consistent and the F-test is overwhelming, but the means run 0.853 → 0.823 → 0.800 → 0.714. Performance **falls** across the exam period, the opposite of the conjecture.

Average GPA falls across the days too, correlating 0.96 with the scores. Better students took the exam earlier, so the day groups differ because the students in them differ.

This is self-selection, from lecture 4, inside a lecture about a test statistic. Students chose their exam day, so day was never randomly assigned. ANOVA tests whether group means differ and nothing more: it is as happy to detect a selection effect as a treatment effect, and it cannot tell you which one it found.